## Compare 2 versions of SCB Myndighetsregistret
* [#59](https://github.com/salgo60/SCB-Wikidata/issues/59)
* [Dumped versions](https://github.com/salgo60/SCB-Wikidata/tree/main/notebook/Myndighetsreg_concat)
   * [Nov 21, 2025 raw](https://raw.githubusercontent.com/salgo60/SCB-Wikidata/800f6c3c4430abd24027b8178e2b7cf5cf545088/notebook/Myndighetsreg_concat/Myndighetsreg_alla.csv)
   * [Jan 12, 2026 raw](https://raw.githubusercontent.com/salgo60/SCB-Wikidata/refs/heads/main/notebook/Myndighetsreg_concat/Myndighetsreg_alla_20260112.csv)
* Notebook [Myndighetsregistret_versions_59](https://github.com/salgo60/SCB-Wikidata/tree/800f6c3c4430abd24027b8178e2b7cf5cf545088/notebook/Myndighetsregistret_versions_59.ipynb)

We skip all organisations without "Organisationsnr" the register has bad quality and no unique identifier

In [1]:
import time

from datetime import datetime

now = datetime.now()
timestamp = now.timestamp()

start_time = time.time()
print("Start:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

Start: 2026-01-13 00:49:41


In [2]:
import os

# Get the current working directory
current_directory = os.getcwd()
print("Current Working Directory:", current_directory)



Current Working Directory: /Users/salgo/Documents/GitHub/SCB-Wikidata/notebook


In [3]:
import pandas as pd
from difflib import SequenceMatcher
from IPython.display import display

v1_url = "https://raw.githubusercontent.com/salgo60/SCB-Wikidata/main/notebook/Myndighetsreg_concat/Myndighetsreg_alla.csv"
v2_url = "https://raw.githubusercontent.com/salgo60/SCB-Wikidata/main/notebook/Myndighetsreg_concat/Myndighetsreg_alla_20260112.csv"

df1 = pd.read_csv(v1_url, dtype=str).fillna("")
df2 = pd.read_csv(v2_url, dtype=str).fillna("")

key = "Organisationsnr"
name_col = "Namn"

# Drop junk rows without OrgNr
df1 = df1[df1[key].str.strip() != ""]
df2 = df2[df2[key].str.strip() != ""]

print("After OrgNr filter:")
print("v1:", len(df1), "v2:", len(df2))


After OrgNr filter:
v1: 347 v2: 341


In [4]:
def canonicalize(df):
    return (
        df
        .sort_values(name_col, key=lambda s: s.str.len(), ascending=False)
        .drop_duplicates(key)
        .set_index(key)
    )

df1c = canonicalize(df1)
df2c = canonicalize(df2)

print("Canonical v1:", len(df1c))
print("Canonical v2:", len(df2c))


Canonical v1: 267
Canonical v2: 261


In [5]:
import pandas as pd
import numpy as np
from IPython.display import display
from difflib import SequenceMatcher

v1_url = "https://raw.githubusercontent.com/salgo60/SCB-Wikidata/main/notebook/Myndighetsreg_concat/Myndighetsreg_alla.csv"
v2_url = "https://raw.githubusercontent.com/salgo60/SCB-Wikidata/main/notebook/Myndighetsreg_concat/Myndighetsreg_alla_20260112.csv"

df1 = pd.read_csv(v1_url, dtype=str).fillna("")
df2 = pd.read_csv(v2_url, dtype=str).fillna("")

print(len(df1), "rows in v1")
print(len(df2), "rows in v2")

key = "Organisationsnr"
name_col = "Namn"

df1 = df1.set_index(key)
df2 = df2.set_index(key)

print("v1:", len(df1), "v2:", len(df2))

ids1 = set(df1.index)
ids2 = set(df2.index)

# Drop junk rows without OrgNr
#df1 = df1[df1[key].str.strip() != ""]
#df2 = df2[df2[key].str.strip() != ""]

#print("Lacks Organisationsnr:", len(df1), "v2:", len(df2))



deleted_ids = ids1 - ids2
added_ids   = ids2 - ids1
common_ids  = ids1 & ids2

print("Deleted:", len(deleted_ids))
print("Added:", len(added_ids))
print("Common:", len(common_ids))



455 rows in v1
449 rows in v2
v1: 455 v2: 449
Deleted: 6
Added: 0
Common: 262


In [6]:
def canonicalize(df, name_col):
    return (
        df
        .reset_index()
        .sort_values(name_col, key=lambda s: s.str.len(), ascending=False)
        .drop_duplicates("Organisationsnr")
        .set_index("Organisationsnr")
    )

df1c = canonicalize(df1.reset_index(), name_col)
df2c = canonicalize(df2.reset_index(), name_col)

print("Canonical v1:", len(df1c))
print("Canonical v2:", len(df2c))


Canonical v1: 268
Canonical v2: 262


In [7]:
ids1 = set(df1c.index)
ids2 = set(df2c.index)

deleted_ids = ids1 - ids2
added_ids   = ids2 - ids1
common_ids  = ids1 & ids2

print("Deleted:", len(deleted_ids))
print("Added:", len(added_ids))
print("Common:", len(common_ids))


Deleted: 6
Added: 0
Common: 262


In [8]:
deleted_ids, added_ids

({'202100-0464',
  '202100-0852',
  '202100-1033',
  '202100-3427',
  '202100-5141',
  '202100-6875'},
 set())

In [9]:
df_deleted_named = (
    df1c
    .loc[list(deleted_ids), [name_col]]
    .reset_index()
    .rename(columns={
        "Organisationsnummer": "OrgNr",
        name_col: "Namn"
    })
    .sort_values("Namn")
)

display(df_deleted_named)

,Organisationsnr,Namn
0,202100-6875,MYNDIGHETEN FÖR ARBETSMILJÖKUNSKAP
2,202100-5141,Myndigheten för stöd till trossamfund
3,202100-0464,STATENS FÖRSVARSHISTORISKA MUSEER
1,202100-1033,STATENS KONSTRÅD
4,202100-0852,STATSKONTORET
5,202100-3427,Statens centrum för arkitektur och design


In [10]:
from difflib import SequenceMatcher

rename_rows = []

for org in common_ids:
    old = df1c.loc[org, name_col]
    new = df2c.loc[org, name_col]

    if old != new:
        sim = SequenceMatcher(None, old.lower(), new.lower()).ratio()
        rename_rows.append([org, old, new, round(sim,3)])

df_renames = pd.DataFrame(
    rename_rows,
    columns=["OrgNr","Old name","New name","Similarity"]
).sort_values("Similarity")

display(df_renames)



,OrgNr,Old name,New name,Similarity
2,202100-5026,EKONOMISTYRNINGSVERKET,Statskontoret,0.400
1,202100-5984,Myndigheten för samhällsskydd och beredskap (MSB),Myndigheten för civilt försvar,0.481
0,202100-2742,FÖRVALTNINGSRÄTTEN I STOCKHOLM OCH MIGRATIONSDOM,KAMMARRÄTTEN I STOCKHOLM OCH MIGRATIONSÖVERDOMST,0.771


In [11]:
reorg_matches = []

for del_id in deleted_ids:
    old_name = df1c.loc[del_id, name_col]

    for add_id in added_ids:
        new_name = df2c.loc[add_id, name_col]
        sim = SequenceMatcher(None, old_name.lower(), new_name.lower()).ratio()

        if sim > 0.6:
            reorg_matches.append([del_id, old_name, add_id, new_name, round(sim,3)])

df_reorg = pd.DataFrame(
    reorg_matches,
    columns=["Old OrgNr","Old name","New OrgNr","New name","Similarity"]
).sort_values("Similarity", ascending=False)

display(df_reorg)


,Old OrgNr,Old name,New OrgNr,New name,Similarity


In [12]:
splits  = df_reorg.groupby("Old OrgNr").filter(lambda x: len(x) > 1)
mergers = df_reorg.groupby("New OrgNr").filter(lambda x: len(x) > 1)

display(splits)
display(mergers)


,Old OrgNr,Old name,New OrgNr,New name,Similarity


,Old OrgNr,Old name,New OrgNr,New name,Similarity


In [13]:
classification = []

for org in deleted_ids:
    if org in splits["Old OrgNr"].values:
        status = "Split"
    elif org in df_reorg["Old OrgNr"].values:
        status = "Merged / Renamed"
    else:
        status = "Closed"

    classification.append([org, df1c.loc[org,name_col], status])

df_deleted_classified = pd.DataFrame(
    classification,
    columns=["OrgNr","Name","Status"]
)

display(df_deleted_classified)


,OrgNr,Name,Status
0,202100-6875,MYNDIGHETEN FÖR ARBETSMILJÖKUNSKAP,Closed
1,202100-1033,STATENS KONSTRÅD,Closed
2,202100-5141,Myndigheten för stöd till trossamfund,Closed
3,202100-0464,STATENS FÖRSVARSHISTORISKA MUSEER,Closed
4,202100-0852,STATSKONTORET,Closed
5,202100-3427,Statens centrum för arkitektur och design,Closed


In [14]:
 # End timer and calculate duration
end_time = time.time()
elapsed_time = end_time - start_time# Bygg audit-lager för den här etappen

# Print current date and total time
print("Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
minutes, seconds = divmod(elapsed_time, 60)
print("Total time elapsed: {:02.0f} minutes {:05.2f} seconds".format(minutes, seconds))


Date: 2026-01-13 00:49:43
Total time elapsed: 00 minutes 01.17 seconds
